<a href="https://colab.research.google.com/github/1FATIMAH1/IT362/blob/main/Phase1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# IT 362 - Phase 1
## Riyadh Rental Data Collection

**Source:** Saudi Open Data Platform - Real Estate General Authority

**Research Question:** In Riyadh, how do neighborhood and property type relate to average annual residential rental prices?

### 1. Import libraries

In [ ]:
import requests
import pandas as pd
from bs4 import BeautifulSoup
import os

### 2. Collect unstructured data
We collect a public text page about real estate indicators from REGA and save the original HTML file.

In [ ]:
page_url = 'https://rega.gov.sa/en/rega/rega-terminologies/digital-real-estate-indicators/'
page_response = requests.get(page_url)
print('Status code:', page_response.status_code)

if page_response.ok:
    with open('rega_indicators_raw.html', 'wb') as file:
        file.write(page_response.content)

    soup = BeautifulSoup(page_response.text, 'html.parser')
    page_text = soup.get_text(' ', strip=True)
    print(page_text[:500])

Status code: 200
Real Estate General Authority -  Digital Real Estate Indicators Skip to Main Content - Official government website of the Government of the Kingdom of Saudi Arabia How to verify Who are you? Individuals Organizations Official Saudi Government website URL ends with gov.sa Website belongs to an official government organization in the Kingdom of Saudi Arabia always ends withgov.sa Government websites use the HTTPS protocol for encryption and security. Secure websites in the Kingdom of Saudi Arabia 


### 3. Request the dataset resources
The API returns the available files for the selected dataset.

In [ ]:
dataset_id = '2dde7e8c-db79-4aec-be4e-37cef64c1d4d'
api_url = 'https://open.data.gov.sa/data/api/datasets/resources?version=-1&dataset=' + dataset_id
resources = []

try:
    response = requests.get(api_url, timeout=30)
    print('Status code:', response.status_code)

    if response.ok:
        data = response.json()
        resources = data['resources']
        print('Number of resources:', len(resources))
    else:
        print('API request failed.')
except:
    print('API connection failed.')

API connection failed.


### 4. Get the raw CSV file
The code downloads the CSV file. If the website does not respond, upload the official file.

In [ ]:
raw_file = 'riyadh_rental_Q1_2026.csv'
csv_url = ''

for item in resources:
    if item['format'] == 'CSV':
        csv_url = item['downloadUrl']

if not os.path.exists(raw_file) and csv_url != '':
    try:
        file_response = requests.get(csv_url, timeout=30)
        file_type = file_response.headers.get('Content-Type', '')
        if file_response.ok and 'text/html' not in file_type:
            with open(raw_file, 'wb') as file:
                file.write(file_response.content)
        else:
            print('CSV download failed.')
    except:
        print('CSV download failed.')

if not os.path.exists(raw_file):
    from google.colab import files
    uploaded = files.upload()
    raw_file = list(uploaded.keys())[0]

print('Raw file:', raw_file)

Saving Rental indicators in Riyadh region 1st Q 2026.csv to Rental indicators in Riyadh region 1st Q 2026.csv
Raw file: Rental indicators in Riyadh region 1st Q 2026.csv


### 5. Read the CSV file

In [ ]:
df = pd.read_csv(raw_file, encoding='utf-8-sig')
print('Rows and columns:', df.shape)
df.head()

Rows and columns: (3534, 10)


,السنة,الربع,المنطقة,المدينة,الحي,نوع العقار,تصنيف العقار,عدد الصفقات,متوسط الايجار ر.س,متوسط الايجار لكل متر مربع ر.س
0,2026,1,الرياض,مرات,الشفاء,دور,سكني,1,26000.0,173.33
1,2026,1,الرياض,الدوادمي,طيبة,فيلا,سكني,27,12900.0,42.26
2,2026,1,الرياض,الجمش,الحلوه,شقة,سكني,1,17500.0,87.50
3,2026,1,الرياض,المجمعة,الأندلس,دور,سكني,11,14650.0,69.84
4,2026,1,الرياض,الرين,الملك فيصل,محل,تجاري,6,0.0,0.00


### 6. Check the data

In [ ]:
print(df.columns)
print(df.dtypes)
print(df.isnull().sum())

Index(['السنة ', 'الربع', 'المنطقة ', 'المدينة', 'الحي ', 'نوع العقار ',
       'تصنيف العقار ', 'عدد الصفقات', 'متوسط الايجار  ر.س',
       'متوسط الايجار لكل متر مربع ر.س'],
      dtype='object')
السنة                               int64
الربع                               int64
المنطقة                            object
المدينة                            object
الحي                               object
نوع العقار                         object
تصنيف العقار                       object
عدد الصفقات                         int64
متوسط الايجار  ر.س                float64
متوسط الايجار لكل متر مربع ر.س    float64
dtype: object
السنة                             0
الربع                             0
المنطقة                           0
المدينة                           0
الحي                              0
نوع العقار                        0
تصنيف العقار                      0
عدد الصفقات                       0
متوسط الايجار  ر.س                0
متوسط الايجار لكل متر مربع ر.س    0
dtype: i

### 7. Select Riyadh residential data

In [ ]:
df.columns = df.columns.str.strip()

riyadh_df = df[(df['المدينة'] == 'الرياض') & (df['تصنيف العقار'] == 'سكني')]

print('Rows and columns:', riyadh_df.shape)
print('Neighborhoods:', riyadh_df['الحي'].nunique())
print('Property types:', riyadh_df['نوع العقار'].unique())
riyadh_df.head()

Rows and columns: (798, 10)
Neighborhoods: 195
Property types: ['دوبلاكس' 'فيلا' 'دور' 'شقة' 'استديو']


,السنة,الربع,المنطقة,المدينة,الحي,نوع العقار,تصنيف العقار,عدد الصفقات,متوسط الايجار ر.س,متوسط الايجار لكل متر مربع ر.س
7,2026,1,الرياض,الرياض,الشفاء,دوبلاكس,سكني,6,30600.00,159.62
11,2026,1,الرياض,الرياض,سلطانة,فيلا,سكني,13,42636.36,96.14
18,2026,1,الرياض,الرياض,الجزيرة,دور,سكني,140,47007.39,201.90
21,2026,1,الرياض,الرياض,الصحافة,شقة,سكني,1140,45693.19,371.72
24,2026,1,الرياض,الرياض,معكال,دور,سكني,7,15171.42,120.04


### 8. Save the final CSV file

In [ ]:
output_file = 'riyadh_residential_Q1_2026.csv'
riyadh_df.to_csv(output_file, index=False, encoding='utf-8-sig')

print('File saved:', output_file)
print('Saved rows:', len(riyadh_df))

File saved: riyadh_residential_Q1_2026.csv
Saved rows: 798
